# Core Silver Transformations

This section transforms the core customer, order, order-item and product entities.

Silver processing includes:

- explicit schema conversion;
- whitespace and blank-value normalization;
- business-key validation;
- business-rule validation;
- duplicate handling;
- quarantine of invalid records;
- source-to-target reconciliation

In [24]:
import uuid
import time
import builtins
from datetime import datetime, timezone
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField ,StringType, LongType, DoubleType, TimestampType


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 26, Finished, Available, Finished, False)

In [25]:
SILVER_RUN_ID = str(uuid.uuid4())
SILVER_STARTED_AT = datetime.now(timezone.utc)
SILVER_START_PERF = time.perf_counter()

LOAD_TYPE = "FULL"

print("Silver run ID: ", SILVER_RUN_ID)
print("Started at: ", SILVER_STARTED_AT)
print("Load type: ", LOAD_TYPE)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 27, Finished, Available, Finished, False)

Silver run ID:  5998e20a-1345-44d5-9602-d00230f7b944
Started at:  2026-07-29 06:40:19.554607+00:00
Load type:  FULL


# Transformation Helpers

Reusable functions standardize text handling, metadata creation and Delta-table writes.

In [26]:
def clean_string(column_name: str):
    """ Trim whitespace and convert blank strings to null. """
    trimmed_value = F.trim(F.col(column_name))
    return F.when(trimmed_value == "",F.lit(None)).otherwise(trimmed_value)

def add_silver_metadata(dataframe: DataFrame, source_table: str) -> DataFrame:
    """ Add technical lineage metadata to a Silver dataframe. """
    return (dataframe
        .withColumn("_silver_run_id", F.lit(SILVER_RUN_ID))
        .withColumn("_silver_processed_at", F.current_timestamp())
        .withColumn("_silver_source_table", F.lit(source_table))
        .withColumn("_silver_load_type", F.lit(LOAD_TYPE))
    )

def write_delta_table(dataframe: DataFrame, table_name: str) -> None:
    """ Overwrite a managed Delta table during the static full-load project. """
    dataframe.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(table_name)

def build_quarantine_reason(*conditions):
    """ Combine multiple reason expressions into one semicolon-separated field.
    Each argument must return either a reason string or null."""
    return F.concat_ws("; ", *conditions)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 28, Finished, Available, Finished, False)

In [27]:
silver_audit_records = []

def record_silver_audit (
    *,
    source_table: str,
    target_table: str,
    quarantine_table: str,
    source_row_count: int,
    valid_row_count: int,
    quarantine_row_count: int,
    duplicate_rows_removed: int,
    status: str
) -> None:

    silver_audit_records.append({
        "silver_run_id": SILVER_RUN_ID,
        "source_table": source_table,
        "target_table": target_table,
        "quarantine_table": quarantine_table,
        "load_type": LOAD_TYPE,
        "source_row_count": int(source_row_count),
        "valid_row_count": int(valid_row_count),
        "quarantine_row_count": int(quarantine_row_count),
        "duplicate_rows_removed": int(duplicate_rows_removed),
        "reconciled_row_count": int(valid_row_count + quarantine_row_count + duplicate_rows_removed),
        "row_count_difference": int(source_row_count - valid_row_count - quarantine_row_count - duplicate_rows_removed),
        "load_status": status,
        "processed_at_utc": datetime.now(timezone.utc)
    })


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 29, Finished, Available, Finished, False)

# Customers Transformation

Transform `bronze_customers` into a typed and standardized customer dataset.

In [28]:
bronze_customers = spark.table("bronze_customers")

customers_source_count = bronze_customers.count()

print("Bronze customer rows:",customers_source_count)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 30, Finished, Available, Finished, False)

Bronze customer rows: 99441


In [29]:
customers_prepared = (
    bronze_customers
    .select(
        clean_string("customer_id").alias("customer_id"),
        clean_string("customer_unique_id").alias("customer_unique_id"),
        clean_string("customer_zip_code_prefix").cast("int").alias("customer_zip_prefix"),
        F.upper(clean_string("customer_city")).alias("customer_city"),
        F.upper(clean_string("customer_state")).alias("customer_state"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 31, Finished, Available, Finished, False)

In [30]:
customers_flagged = (
    customers_prepared
    .withColumn("_dq_missing_customer_id", F.col("customer_id").isNull())
    .withColumn("_dq_missing_unique_customer_id", F.col("customer_unique_id").isNull())
    .withColumn("_dq_invalid_zip_prefix", F.col("customer_zip_prefix").isNull())
    .withColumn("_dq_missing_state", F.col("customer_state").isNull())
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 32, Finished, Available, Finished, False)

In [31]:
customers_flagged = (
    customers_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_customer_id"), F.lit("MISSING_CUSTOMER_ID")),
            F.when(F.col("_dq_missing_unique_customer_id"), F.lit("MISSING_CUSTOMER_UNIQUE_ID")),
            F.when(F.col("_dq_invalid_zip_prefix"), F.lit("INVALID_CUSTOMER_ZIP_PREFIX")),
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 33, Finished, Available, Finished, False)

In [32]:
quarantine_customers = (
    customers_flagged
    .filter(
        F.col("_dq_missing_customer_id")
        | F.col("_dq_missing_unique_customer_id")
        | F.col("_dq_invalid_zip_prefix")
    )
)

valid_customers_pre_dedup = (
    customers_flagged
    .filter(
        ~(
            F.col("_dq_missing_customer_id")
            | F.col("_dq_missing_unique_customer_id")
            | F.col("_dq_invalid_zip_prefix")
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 34, Finished, Available, Finished, False)

In [33]:
customer_dedup_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

customers_ranked = (
    valid_customers_pre_dedup
    .withColumn("_duplicate_rank", F.row_number().over(customer_dedup_window))
)

duplicate_customer_rows = (
    customers_ranked
    .filter(F.col("_duplicate_rank") > 1).count()
)

silver_customers = (
    customers_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank", "_quarantine_reason")
)

silver_customers = add_silver_metadata(silver_customers, "bronze_customers")
quarantine_customers = add_silver_metadata(quarantine_customers, "bronze_customers")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 35, Finished, Available, Finished, False)

In [34]:
write_delta_table(silver_customers, "silver_customers")
write_delta_table(quarantine_customers, "quarantine_customers")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 36, Finished, Available, Finished, False)

In [35]:
customers_valid_count = spark.table("silver_customers").count()
customers_quarantine_count = spark.table("quarantine_customers").count()
customers_reconciliation_difference = customers_source_count - customers_valid_count - customers_quarantine_count - duplicate_customer_rows

customers_status = (
    "SUCCESS"
    if customers_reconciliation_difference == 0
    else "FAILED"
)

record_silver_audit(
    source_table = "bronze_customers",
    target_table = "silver_customers",
    quarantine_table = "quarantine_customers",
    source_row_count = customers_source_count,
    valid_row_count = customers_valid_count,
    quarantine_row_count = customers_quarantine_count,
    duplicate_rows_removed = duplicate_customer_rows,
    status = customers_status
)

print(
    "Customers:",
    {
        "source": customers_source_count,
        "valid": customers_valid_count,
        "quarantine": customers_quarantine_count,
        "duplicates_removed": duplicate_customer_rows,
        "status": customers_status
    }
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 37, Finished, Available, Finished, False)

Customers: {'source': 99441, 'valid': 99441, 'quarantine': 0, 'duplicates_removed': 0, 'status': 'SUCCESS'}


# Orders Transformation

Create a typed order table with lifecycle and delivery-quality features.

In [36]:
bronze_orders = spark.table("bronze_orders")

orders_source_count = bronze_orders.count()

orders_prepared = (
    bronze_orders
    .select(
        clean_string("order_id").alias("order_id"),
        clean_string("customer_id").alias("customer_id"),
        F.lower(clean_string("order_status")).alias("order_status"),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.to_timestamp("order_approved_at").alias("order_approved_ts"),
        F.to_timestamp("order_delivered_carrier_date").alias("carrier_handover_ts"),
        F.to_timestamp("order_delivered_customer_date").alias("delivered_customer_ts"),
        F.to_timestamp("order_estimated_delivery_date").alias("estimated_delivery_ts"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 38, Finished, Available, Finished, False)

In [37]:
VALID_ORDER_STATUSES = [
    "approved", "canceled", "created", "delivered", "invoiced", "processing","shipped", "unavailable"
]

orders_flagged = (
    orders_prepared
    .withColumn("_dq_missing_order_id", F.col("order_id").isNull())
    .withColumn("_dq_missing_customer_id", F.col("customer_id").isNull())
    .withColumn("_dq_invalid_order_status", F.col("order_status").isNull() | ~F.col("order_status").isin(VALID_ORDER_STATUSES))
    .withColumn("_dq_missing_purchase_timestamp", F.col("order_purchase_ts").isNull())
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 39, Finished, Available, Finished, False)

In [38]:
orders_flagged = (
    orders_flagged
    .withColumn(
        "_dq_approval_before_purchase",
        F.col("order_approved_ts").isNotNull() & (F.col("order_approved_ts")<F.col("order_purchase_ts"))
    )
    .withColumn(
        "_dq_carrier_before_approval",
        (
            F.col("carrier_handover_ts").isNotNull()
            & F.col("order_approved_ts").isNotNull()
            & (
                F.col("carrier_handover_ts")
                < F.col("order_approved_ts")
            )
        )
    )
    .withColumn(
        "_dq_delivery_before_purchase",
        (
            F.col("delivered_customer_ts").isNotNull()
            & (
                F.col("delivered_customer_ts")
                < F.col("order_purchase_ts")
            )
        )
    )
    .withColumn(
        "_dq_delivered_status_missing_delivery_ts",
        (
            F.col("order_status") == "delivered"
        )
        & F.col(
            "delivered_customer_ts"
        ).isNull()
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 40, Finished, Available, Finished, False)

In [39]:
orders_flagged = (
    orders_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_order_id"), F.lit("MISSING_ORDER_ID")),
            F.when(F.col("_dq_missing_customer_id"), F.lit("MISSING_CUSTOMER_ID")),
            F.when(F.col("_dq_invalid_order_status"), F.lit("INVALID_ORDER_STATUS")),
            F.when(F.col("_dq_missing_purchase_timestamp"),F.lit("MISSING_OR_INVALID_PURCHASE_TIMESTAMP"))
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 41, Finished, Available, Finished, False)

In [40]:
orders_critical_failure = (
    F.col("_dq_missing_order_id")
    | F.col("_dq_missing_customer_id")
    | F.col("_dq_invalid_order_status")
    | F.col("_dq_missing_purchase_timestamp")
)

quarantine_orders = orders_flagged.filter(orders_critical_failure)
valid_orders_pre_dedup = orders_flagged.filter(~orders_critical_failure)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 42, Finished, Available, Finished, False)

In [41]:
order_dedup_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc())
)

orders_ranked = (
    valid_orders_pre_dedup
    .withColumn("_duplicate_rank",F.row_number().over(order_dedup_window))
)

duplicate_order_rows = orders_ranked.filter(F.col("_duplicate_rank") > 1).count()

silver_orders = orders_ranked.filter(F.col("_duplicate_rank") == 1).drop("_duplicate_rank","_duplicate_reason")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 43, Finished, Available, Finished, False)

In [42]:
silver_orders = (
    silver_orders
    .withColumn("order_purchase_date",F.to_date("order_purchase_ts"))
    .withColumn("purchase_year",F.year("order_purchase_ts"))
    .withColumn("purchase_month",F.month("order_purchase_ts"))
    .withColumn("purchase_hour",F.hour("order_purchase_ts"))
    .withColumn("purchase_weekday_number",F.dayofweek("order_purchase_ts"))
    .withColumn("purchase_weekday_name",F.date_format("order_purchase_ts","EEEE"))
    .withColumn("is_weekend_purchase",F.when(F.dayofweek("order_purchase_ts").isin(1, 7),True).otherwise(False))
)

silver_orders = (
    silver_orders
    .withColumn(
        "approval_hours",
        (
            F.unix_timestamp(
                "order_approved_ts"
            )
            - F.unix_timestamp(
                "order_purchase_ts"
            )
        ) / F.lit(3600.0)
    )
    .withColumn(
        "carrier_handover_days",
        F.datediff(
            "carrier_handover_ts",
            "order_approved_ts"
        )
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "delivered_customer_ts",
            "order_purchase_ts"
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            "delivered_customer_ts",
            "estimated_delivery_ts"
        )
    )
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("delivered_customer_ts").isNotNull()
            & F.col("estimated_delivery_ts").isNotNull()
            & (
                F.col("delivered_customer_ts")
                > F.col("estimated_delivery_ts")
            ),
            True
        )
        .when(
            F.col("delivered_customer_ts").isNotNull()
            & F.col("estimated_delivery_ts").isNotNull(),
            False
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 44, Finished, Available, Finished, False)

In [43]:
silver_orders = add_silver_metadata(silver_orders, "bronze_orders")
quarantine_orders = add_silver_metadata(quarantine_orders, "bronze_orders")

write_delta_table(silver_orders, "silver_orders")
write_delta_table(quarantine_orders, "quarantine_orders")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 45, Finished, Available, Finished, False)

In [45]:
orders_valid_count = spark.table("silver_orders").count()
orders_quarantine_count = spark.table("quarantine_orders").count()
orders_difference = orders_source_count - orders_valid_count - orders_quarantine_count - duplicate_order_rows
orders_status = "SUCCESS" if orders_difference == 0 else "FAILED"

record_silver_audit(
    source_table = "bronze_orders",
    target_table = "silver_orders",
    quarantine_table = "quarantine_orders",
    source_row_count = orders_source_count,
    valid_row_count = orders_valid_count,
    quarantine_row_count = orders_quarantine_count,
    duplicate_rows_removed = duplicate_order_rows,
    status = orders_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 47, Finished, Available, Finished, False)

# Order-Item Transformation

Create a typed order-item table at one row per order and item sequence.

In [48]:
bronze_order_items = spark.table("bronze_order_items")

order_items_source_count = bronze_order_items.count()

order_items_prepared = (
    bronze_order_items
    .select(
        clean_string("order_id").alias("order_id"),
        clean_string("order_item_id").cast("int").alias("order_item_id"),
        clean_string("product_id").alias("product_id"),
        clean_string("seller_id").alias("seller_id"),
        F.to_timestamp("shipping_limit_date").alias("shipping_limit_ts"),
        clean_string("price").cast("decimal(18,2)").alias("item_price"),
        clean_string("freight_value").cast("decimal(18,2)").alias("freight_value"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 52, Finished, Available, Finished, False)

In [49]:
order_items_flagged = (
    order_items_prepared
    .withColumn("_dq_missing_order_id",F.col("order_id").isNull())
    .withColumn("_dq_invalid_order_item_id",
        F.col("order_item_id").isNull() | (F.col("order_item_id") <= 0)
    )
    .withColumn("_dq_missing_product_id",F.col("product_id").isNull())
    .withColumn("_dq_missing_seller_id",F.col("seller_id").isNull())
    .withColumn("_dq_invalid_item_price",
        F.col("item_price").isNull() | (F.col("item_price") < 0)
    )
    .withColumn("_dq_invalid_freight_value",
        F.col("freight_value").isNull() | (F.col("freight_value") < 0)
    )
)

order_items_critical_failure = (
    F.col("_dq_missing_order_id")
    | F.col("_dq_invalid_order_item_id")
    | F.col("_dq_missing_product_id")
    | F.col("_dq_missing_seller_id")
    | F.col("_dq_invalid_item_price")
    | F.col("_dq_invalid_freight_value")
)

order_items_flagged = (
    order_items_flagged
    .withColumn(
        "_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_order_id"),F.lit("MISSING_ORDER_ID")),
            F.when(F.col("_dq_invalid_order_item_id"),F.lit("INVALID_ORDER_ITEM_ID")),
            F.when(F.col("_dq_missing_product_id"),F.lit("MISSING_PRODUCT_ID")),
            F.when(F.col("_dq_missing_seller_id"),F.lit("MISSING_SELLER_ID")),
            F.when(F.col("_dq_invalid_item_price"),F.lit("INVALID_ITEM_PRICE")),
            F.when(F.col("_dq_invalid_freight_value"),F.lit("INVALID_FREIGHT_VALUE"))
        )
    )
)

quarantine_order_items = order_items_flagged.filter(order_items_critical_failure)

valid_order_items_pre_dedup = order_items_flagged.filter(~order_items_critical_failure)


StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 53, Finished, Available, Finished, False)

In [50]:
order_item_dedup_window = (
    Window
    .partitionBy("order_id","order_item_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

order_items_ranked = (
    valid_order_items_pre_dedup
    .withColumn("_duplicate_rank",F.row_number().over(order_item_dedup_window))
)

duplicate_order_item_rows = order_items_ranked.filter(F.col("_duplicate_rank") > 1).count()

silver_order_items = (
    order_items_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank","_quarantine_reason")
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 54, Finished, Available, Finished, False)

In [51]:
silver_order_items = (
    silver_order_items
    .withColumn("item_total_value",F.col("item_price") + F.col("freight_value"))
    .withColumn("freight_to_price_ratio",
        F.when(
            F.col("item_price") > 0,
            (F.col("freight_value") / F.col("item_price")).cast("decimal(18,6)")
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 55, Finished, Available, Finished, False)

In [52]:
silver_order_items = add_silver_metadata(silver_order_items,"bronze_order_items")
quarantine_order_items = add_silver_metadata(quarantine_order_items,"bronze_order_items")

write_delta_table(silver_order_items,"silver_order_items")
write_delta_table(quarantine_order_items,"quarantine_order_items")

order_items_valid_count = spark.table("silver_order_items").count()
order_items_quarantine_count = spark.table("quarantine_order_items").count()
order_items_difference = order_items_source_count - order_items_valid_count - order_items_quarantine_count - duplicate_order_item_rows
order_items_status = "SUCCESS" if order_items_difference == 0 else "FAILED"

record_silver_audit(
    source_table="bronze_order_items",
    target_table="silver_order_items",
    quarantine_table="quarantine_order_items",
    source_row_count=order_items_source_count,
    valid_row_count=order_items_valid_count,
    quarantine_row_count=order_items_quarantine_count,
    duplicate_rows_removed=duplicate_order_item_rows,
    status=order_items_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 56, Finished, Available, Finished, False)

# Products Transformation

Create a typed product table with standardized physical attributes and quality flags.

In [54]:
bronze_products = spark.table("bronze_products")

products_source_count = bronze_products.count()

products_prepared = (
    bronze_products
    .select(
        clean_string("product_id").alias("product_id"),
        F.lower(clean_string("product_category_name")).alias("product_category_name_pt"),
        clean_string("product_name_lenght").cast("int").alias("product_name_length"),
        clean_string("product_description_lenght").cast("int").alias("product_description_length"),
        clean_string("product_photos_qty").cast("int").alias("product_photo_count"),
        clean_string("product_weight_g").cast("double").alias("product_weight_g"),
        clean_string("product_length_cm").cast("double").alias("product_length_cm"),
        clean_string("product_height_cm").cast("double").alias("product_height_cm"),
        clean_string("product_width_cm").cast("double").alias("product_width_cm"),
        "_bronze_record_hash",
        "_bronze_run_id",
        "_bronze_ingested_at"
    )
)

products_flagged = (
    products_prepared
    .withColumn(
        "_dq_missing_product_id",
        F.col("product_id").isNull()
    )
    .withColumn(
        "_dq_missing_category",
        F.col(
            "product_category_name_pt"
        ).isNull()
    )
    .withColumn(
        "_dq_invalid_weight",
        (
            F.col("product_weight_g").isNotNull()
            & (
                F.col("product_weight_g") < 0
            )
        )
    )
    .withColumn(
        "_dq_invalid_dimensions",
        (
            (
                F.col("product_length_cm").isNotNull()
                & (
                    F.col("product_length_cm") < 0
                )
            )
            | (
                F.col("product_height_cm").isNotNull()
                & (
                    F.col("product_height_cm") < 0
                )
            )
            | (
                F.col("product_width_cm").isNotNull()
                & (
                    F.col("product_width_cm") < 0
                )
            )
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 58, Finished, Available, Finished, False)

In [55]:
products_critical_failure = (
    F.col("_dq_missing_product_id")
    | F.col("_dq_invalid_weight")
    | F.col("_dq_invalid_dimensions")
)

products_flagged = (
    products_flagged
    .withColumn("_quarantine_reason",
        build_quarantine_reason(
            F.when(F.col("_dq_missing_product_id"),F.lit("MISSING_PRODUCT_ID")),
            F.when(F.col("_dq_invalid_weight"),F.lit("INVALID_PRODUCT_WEIGHT")),
            F.when(F.col("_dq_invalid_dimensions"),F.lit("INVALID_PRODUCT_DIMENSIONS"))
        )
    )
)

quarantine_products = products_flagged.filter(products_critical_failure)
valid_products_pre_dedup = products_flagged.filter(~products_critical_failure)

product_dedup_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("_bronze_ingested_at").desc_nulls_last(),
        F.col("_bronze_record_hash").asc()
    )
)

products_ranked = (
    valid_products_pre_dedup
    .withColumn("_duplicate_rank",F.row_number().over(product_dedup_window))
)

duplicate_product_rows = products_ranked.filter(F.col("_duplicate_rank") > 1).count()

silver_products = (
    products_ranked
    .filter(F.col("_duplicate_rank") == 1)
    .drop("_duplicate_rank","_quarantine_reason")
)

silver_products = (
    silver_products
    .withColumn(
        "product_volume_cm3",
        F.when(
            F.col("product_length_cm").isNotNull()
            & F.col("product_height_cm").isNotNull()
            & F.col("product_width_cm").isNotNull(),
            F.col("product_length_cm")
            * F.col("product_height_cm")
            * F.col("product_width_cm")
        )
    )
    .withColumn(
        "product_density_g_per_cm3",
        F.when(
            F.col("product_volume_cm3") > 0,
            F.col("product_weight_g")
            / F.col("product_volume_cm3")
        )
    )
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 59, Finished, Available, Finished, False)

In [56]:
silver_products = add_silver_metadata(silver_products,"bronze_products")
quarantine_products = add_silver_metadata(quarantine_products,"bronze_products")

write_delta_table(silver_products,"silver_products")
write_delta_table(quarantine_products,"quarantine_products")

products_valid_count = spark.table("silver_products").count()
products_quarantine_count = spark.table("quarantine_products").count()
products_difference = products_source_count - products_valid_count - products_quarantine_count - duplicate_product_rows
products_status = "SUCCESS" if products_difference == 0 else "FAILED"

record_silver_audit(
    source_table="bronze_products",
    target_table="silver_products",
    quarantine_table="quarantine_products",
    source_row_count=products_source_count,
    valid_row_count=products_valid_count,
    quarantine_row_count=products_quarantine_count,
    duplicate_rows_removed=duplicate_product_rows,
    status=products_status
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 60, Finished, Available, Finished, False)

In [59]:
""" Saving Audit Table """

silver_audit_schema = StructType([
    StructField("silver_run_id", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("quarantine_table", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("source_row_count", LongType(), False),
    StructField("valid_row_count", LongType(), False),
    StructField("quarantine_row_count", LongType(), False),
    StructField("duplicate_rows_removed", LongType(), False),
    StructField("reconciled_row_count", LongType(), False),
    StructField("row_count_difference", LongType(), False),
    StructField("load_status", StringType(), False),
    StructField("processed_at_utc", TimestampType(), False)
])

silver_audit_df = spark.createDataFrame(silver_audit_records,schema=silver_audit_schema)

silver_audit_df.write.mode("append").format("delta").saveAsTable("audit_silver_load")

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 63, Finished, Available, Finished, False)

In [67]:
display(
    spark.table("audit_silver_load")
    .limit(20)
)

StatementMeta(, f5fd11cd-6cde-49ab-956a-6b267feb807d, 71, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7bbef786-31b6-4b0c-8022-4e36b675f5d3)